# Fine-tune PaddleOCR recognition on synthetic dot-peen lines (Stage 3)

**Data:** the attached Kaggle dataset `dotpeen-synth` (synthetic train + validation only).
The real photos are **not** uploaded and never used here: after download, each epoch's
checkpoint is scored on them on the Mac (`ml/scripts/select_checkpoint.py`).

Settings: *Accelerator* = **GPU (T4 or P100)**, *Internet* = **On** (to install PaddlePaddle,
download PaddleOCR v3.7.0 and the pretrained weights). Run all cells in order.

In [ ]:
# 1. PaddlePaddle GPU 3.3.1 (CUDA 12.6 build) and PaddleOCR v3.7.0 source (Apache-2.0)
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!pip install paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!git clone --depth 1 --branch v3.7.0 https://github.com/PaddlePaddle/PaddleOCR.git /kaggle/working/PaddleOCR
!pip install -r /kaggle/working/PaddleOCR/requirements.txt
# Checked in a separate process, so this notebook does not keep GPU memory for itself.
!python -c "import paddle; print(paddle.__version__, 'CUDA:', paddle.is_compiled_with_cuda()); paddle.utils.run_check()"

In [ ]:
# 2. Find the uploaded dataset (Kaggle unpacks it under /kaggle/input/<name>/)
import glob, os
lists = glob.glob('/kaggle/input/**/synth/train_list.txt', recursive=True)
assert lists, 'dataset not attached: Add Input -> your dotpeen-synth dataset'
DATA = os.path.dirname(lists[0])
CODE = os.path.join(os.path.dirname(DATA), 'code')
print(DATA, len(open(lists[0]).read().splitlines()), 'training lines')
print(open(os.path.join(DATA, 'stats.json')).read()[:400])

In [ ]:
# 3. Train: a checkpoint after EVERY epoch (output/epochs/), synthetic validation loss and
#    exact match per epoch (output/history.csv); early stopping on synthetic validation loss.
# Each log line is shown as soon as it is written (unbuffered), also saved to output/train.log.
import os, subprocess, sys
BATCH = 64  # 128 ran out of memory on a T4 (15 GB); lower to 32 if it still does
cmd = [sys.executable, "-u", f"{CODE}/train_rec.py",
       "--paddleocr", "/kaggle/working/PaddleOCR", "--data", DATA,
       "--out", "/kaggle/working/output", "--config", f"{CODE}/finetune_rec.yml", "--workers", "4", "--batch", str(BATCH)]
proc = subprocess.Popen(cmd, cwd="/kaggle/working", stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
for line in proc.stdout:
    if "Warning" in line or "warnings.warn" in line:
        continue  # library warnings; everything else is shown
    print(line, end="", flush=True)
assert proc.wait() == 0, f"training failed (exit code {proc.returncode}); see output above"


In [ ]:
# 4. Curves and the synthetic validation history (the checkpoint is chosen on the Mac)
import pandas as pd, json
from IPython.display import Image, display
print('lowest synthetic val loss:', json.load(open('/kaggle/working/output/best_synth.json')))
display(pd.read_csv('/kaggle/working/output/history.csv'))
display(Image('/kaggle/working/output/curves.png'))


In [ ]:
# 5. Pack the results for download (Output tab -> result.zip): every epoch's checkpoint
#    (~22 MB each), history, curves, log, config, and the full last checkpoint.
!cd /kaggle/working/output && zip -q -r /kaggle/working/result.zip epochs history.csv curves.png config_used.yml train.log best_synth.json best_synth.pdparams last.pdparams
!ls -la /kaggle/working/result.zip
